# Kaggle runner — Trustworthy Skin Lesion Triage

All installs, data and training happen here on Kaggle, not on the laptop. The laptop holds only the code (git).

**One-time notebook settings (right-hand panel):**
1. Accelerator: **GPU T4 x2** (or P100).
2. Internet: **On** (needed for `git clone`, `pip`, pretrained weights).
3. Add Input → Datasets → `kmader/skin-cancer-mnist-ham10000`.
4. If the GitHub repo is private: Add-ons → Secrets → add `GITHUB_TOKEN` (a fine-grained read-only token for this repo).

Then set `REPO_URL` and `PHASE_COMMANDS` below and **Run All**. Download `results.zip` from the Output tab and unzip it into the repo's `results/` on the laptop.

In [ ]:
REPO_URL = "https://github.com/diyajain0902/Skin-Lesion-Classification.git"
BRANCH = "phase-1-setup"   # branch being verified; use "main" once merged

# Commands for the phase being run (each is run from the repo root).
PHASE_COMMANDS = [
    "python -m src.check_env",
    "python -m pytest",
]

In [ ]:
import os, subprocess

# Use a token for private repos; it is read from Kaggle Secrets and never printed.
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
except Exception:
    pass  # public repo, no secret configured

os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle already ships a CUDA build of torch/torchvision; keep it and install only the rest.
!grep -vE '^(torch|torchvision)([<>=]|$)' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt

In [ ]:
# Run the phase. Stops at the first failing command so a broken step is never hidden.
for cmd in PHASE_COMMANDS:
    print(f"\n$ {cmd}", flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0:
        raise SystemExit(f"FAILED (exit {rc}): {cmd}")

In [ ]:
# Package plain-file results (and fixed splits, once they exist) for download.
!cd /kaggle/working/repo && zip -qr /kaggle/working/results.zip results splits 2>/dev/null; ls -lh /kaggle/working/results.zip